In [ ]:
import pandas as pd
from collections import Counter
import re
import plotly.express as px
import plotly.io as pio
pio.templates.default ="plotly_white"

In [ ]:
queries_df=pd.read_csv("C:/Users/udayi/Downloads/Phone Link/Queries.csv")

In [ ]:
queries_df['CTR']= queries_df['CTR'].str.rstrip('%').astype('float')/100

In [ ]:
def clean_and_split(query):
    words=re.findall(r'\b[a-zA-Z]+\b',query.lower())
    return words

In [ ]:
word_counts=Counter()
for query in queries_df['Top queries']:
    word_counts.update(clean_and_split(query))

In [ ]:
word_freq_count=pd.DataFrame(word_counts.most_common(20),columns=['words','Frequency'])

In [ ]:
print(word_freq_count.head())

In [ ]:
fig=px.bar(word_freq_count,x='words',y='Frequency',title='top most 20 common word in the search queries')
fig.show()

In [ ]:
top_queries_clicks_vis=queries_df.nlargest(10,'Clicks')[['Top queries','Clicks']]
top_queries_impression_vis=queries_df.nlargest(10,'Impressions')[['Top queries','Impressions']]

In [ ]:
fig_clicks=px.bar(top_queries_clicks_vis,x='Top queries',y='Clicks',title='Top queries by Clicks')
fig_Impressions=px.bar(top_queries_impression_vis,x='Top queries',y='Impressions',title='Top queries by Impressions')
fig_clicks.show()
fig_Impressions.show()

In [ ]:
top_ctr_vis=queries_df.nlargest(10,'CTR')[['Top queries','CTR']]
bottom_ctr_vis=queries_df.nsmallest(10,'CTR')[['Top queries','CTR']]

In [ ]:
fig_top_ctr=px.bar(top_ctr_vis,x='Top queries',y='CTR',title='Top queries by CTR')
fig_bottom_ctr=px.bar(bottom_ctr_vis,x='Top queries',y='CTR',title=' bottom queries by CTR')
fig_top_ctr.show()
fig_bottom_ctr.show()

In [ ]:
correlation_matrix=queries_df[['Clicks','Impressions','CTR','Position']].corr()
fig_corr=px.imshow(correlation_matrix,text_auto=True,title='correlation matrix')
fig_corr.show()


In [ ]:
from sklearn.ensemble import IsolationForest
features=queries_df[['Clicks','Impressions','CTR','Position']]
iso_forest=IsolationForest(n_estimators=100,contamination=0.01)
iso_forest.fit(features)
queries_df['anomaly']=iso_forest.predict(features)
anomalies=queries_df[queries_df['anomaly']== 1]
print(anomalies[['Top queries','Clicks','Impressions','CTR','Position']])


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
normal=queries_df[queries_df['anomaly']== -1]
anomalies=queries_df[queries_df['anomaly']==1]
plt.figure(figsize=(8,6))
plt.scatter(normal['Impressions'],normal['Clicks'],color='blue',label='Normal',alpha=0.6,s=60)
plt.scatter(anomalies['Impressions'],anomalies['Clicks'],color='red',label='anomaly',alpha=0.8,s=70)
plt.title('Anomaly Detection in Search Queries(IsolationForest)')
plt.xlabel('Impressions')
plt.ylabel('Clicks')
plt.legend()
plt.grid(True,linestyle='--',alpha=0.5)
plt.show()